In [1]:
 from google.colab import drive
 drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q pydicom nibabel numpy pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.1 MB/s eta 0:00:00


In [3]:



import os
import math
import csv
from datetime import datetime, timedelta

import numpy as np
import pydicom
import nibabel as nib
import pandas as pd


def safe_float(x, default=None):
    try:
        if x is None or x == "":
            return default
        return float(x)
    except Exception:
        return default


def parse_dicom_tm(tm_str):
    if not tm_str:
        return None
    s = str(tm_str).strip()
    frac = 0
    if "." in s:
        base, frac_str = s.split(".", 1)
        frac = int((frac_str + "000000")[:6])
    else:
        base = s
    base = base.ljust(6, "0")
    try:
        h = int(base[0:2])
        m = int(base[2:4])
        sec = int(base[4:6])
        return h, m, sec, frac
    except Exception:
        return None


def parse_dicom_da(da_str):
    if not da_str:
        return None
    try:
        return datetime.strptime(str(da_str), "%Y%m%d")
    except Exception:
        return None


def combine_da_tm(date_str, time_str):
    d = parse_dicom_da(date_str)
    t = parse_dicom_tm(time_str)
    if d is None or t is None:
        return None
    return d.replace(hour=t[0], minute=t[1], second=t[2], microsecond=t[3])


def parse_dicom_dt(dt_str):
    if not dt_str:
        return None
    s = str(dt_str).strip()
    for sign in ["+", "-"]:
        if len(s) > 14 and sign in s[14:]:
            s = s.split(sign, 1)[0]
            break
    for fmt in ["%Y%m%d%H%M%S.%f", "%Y%m%d%H%M%S", "%Y%m%d%H%M", "%Y%m%d"]:
        try:
            return datetime.strptime(s, fmt)
        except Exception:
            pass
    return None


def find_case_dirs(root_dir):
    return [
        os.path.join(root_dir, name)
        for name in sorted(os.listdir(root_dir))
        if os.path.isdir(os.path.join(root_dir, name))
    ]


def find_first_nii_gz(mask_dir):
    files = sorted([f for f in os.listdir(mask_dir) if f.endswith(".nii.gz")])
    if not files:
        raise FileNotFoundError(f"No .nii.gz found in {mask_dir}")
    return os.path.join(mask_dir, files[0])


def find_dicom_files(pt_dir):
    dicoms = []
    for root, _, files in os.walk(pt_dir):
        for f in files:
            path = os.path.join(root, f)
            try:
                ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
                if getattr(ds, "Modality", None) == "PT":
                    dicoms.append(path)
            except Exception:
                pass
    if not dicoms:
        raise FileNotFoundError(f"No PET DICOM files found in {pt_dir}")
    return dicoms


def sort_pet_slices(datasets):
    def key(ds):
        acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
        if acq_dt is None:
            acq_dt = combine_da_tm(
                getattr(ds, "AcquisitionDate", None),
                getattr(ds, "AcquisitionTime", None),
            )
        z = 0.0
        ipp = getattr(ds, "ImagePositionPatient", None)
        if ipp is not None and len(ipp) >= 3:
            z = float(ipp[2])
        inst = safe_float(getattr(ds, "InstanceNumber", None), 0)
        return (acq_dt or datetime.min, z, inst)
    return sorted(datasets, key=key)


def load_pet_series(pt_dir):
    dicom_files = find_dicom_files(pt_dir)
    datasets = [pydicom.dcmread(p, force=True) for p in dicom_files]
    datasets = sort_pet_slices(datasets)

    arrays, slopes, intercepts = [], [], []
    for ds in datasets:
        pixel = ds.pixel_array.astype(np.float64)
        if pixel.ndim != 2:
            raise NotImplementedError("Enhanced multiframe PET is not supported.")
        arrays.append(pixel)
        slopes.append(safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0))
        intercepts.append(safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0))

    volume = np.stack(arrays, axis=0)
    return (
        datasets,
        volume,
        np.asarray(slopes, dtype=np.float64),
        np.asarray(intercepts, dtype=np.float64),
    )


def apply_rescale(volume, slopes, intercepts):
    out = np.empty_like(volume, dtype=np.float64)
    for z in range(volume.shape[0]):
        out[z] = volume[z] * slopes[z] + intercepts[z]
    return out


def get_radiopharm_item(ds):
    seq = getattr(ds, "RadiopharmaceuticalInformationSequence", None)
    if not seq or len(seq) == 0:
        raise ValueError("Missing RadiopharmaceuticalInformationSequence")
    return seq[0]


def get_weight_kg(ds):
    w = safe_float(getattr(ds, "PatientWeight", None))
    if w is None or w == 0:
        raise ValueError("Missing or zero PatientWeight")
    if w > 1000:
        w /= 1000.0
    return w


def get_weight_g(ds):
    return get_weight_kg(ds) * 1000.0


def get_height_m(ds):
    h = safe_float(getattr(ds, "PatientSize", None))
    if h is None or h <= 0:
        raise ValueError("Missing or invalid PatientSize")
    return h


def get_height_cm(ds):
    return get_height_m(ds) * 100.0


def get_patient_sex(ds):
    sex = str(getattr(ds, "PatientSex", "O") or "O").upper()
    return sex if sex in ["M", "F", "O"] else "O"


def get_total_dose_bq(ds):
    rp = get_radiopharm_item(ds)
    dose = safe_float(getattr(rp, "RadionuclideTotalDose", None))
    if dose is None or dose <= 0:
        raise ValueError("Missing or invalid RadionuclideTotalDose")
    if dose < 1e7:
        dose *= 1e6
    return dose


def get_half_life_s(ds):
    rp = get_radiopharm_item(ds)
    hl = safe_float(getattr(rp, "RadionuclideHalfLife", None))
    if hl is None or hl <= 0:
        raise ValueError("Missing or invalid RadionuclideHalfLife")
    return hl


def get_radiopharm_start_datetime(ds):
    rp = get_radiopharm_item(ds)
    dt = parse_dicom_dt(getattr(rp, "RadiopharmaceuticalStartDateTime", None))
    if dt is not None:
        return dt

    start_time = getattr(rp, "RadiopharmaceuticalStartTime", None)
    if not start_time:
        raise ValueError("Missing radiopharmaceutical start date/time")

    acq_date = getattr(ds, "AcquisitionDate", None) or getattr(ds, "SeriesDate", None)
    acq_time = getattr(ds, "AcquisitionTime", None) or getattr(ds, "SeriesTime", None)

    start_dt = combine_da_tm(acq_date, start_time)
    acq_dt = combine_da_tm(acq_date, acq_time)

    if start_dt is None:
        raise ValueError("Could not construct radiopharmaceutical start datetime")

    if acq_dt is not None and start_dt > acq_dt:
        start_dt -= timedelta(days=1)

    return start_dt


def average_count_rate_time_s(half_life_s, frame_duration_s):
    lam = math.log(2.0) / half_life_s
    return (1.0 / lam) * math.log((lam * frame_duration_s) / (1.0 - math.exp(-lam * frame_duration_s)))


def get_reference_time_start(ds):
    half_life_s = get_half_life_s(ds)
    frame_duration_ms = safe_float(getattr(ds, "ActualFrameDuration", None))
    frame_ref_ms = safe_float(getattr(ds, "FrameReferenceTime", None))

    acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
    if acq_dt is None:
        acq_dt = combine_da_tm(getattr(ds, "AcquisitionDate", None), getattr(ds, "AcquisitionTime", None))

    if acq_dt is None:
        raise ValueError("Missing acquisition datetime")
    if frame_duration_ms is None or frame_ref_ms is None:
        raise ValueError("Missing ActualFrameDuration or FrameReferenceTime")

    frame_duration_s = frame_duration_ms / 1000.0
    frame_ref_s = frame_ref_ms / 1000.0
    tave_s = average_count_rate_time_s(half_life_s, frame_duration_s)
    return acq_dt + timedelta(seconds=(tave_s - frame_ref_s))


def lbm_james128_kg(weight_kg, height_cm, sex):
    male = 1.10 * weight_kg - 128.0 * (weight_kg / height_cm) ** 2
    female = 1.07 * weight_kg - 148.0 * (weight_kg / height_cm) ** 2
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


def lbm_morgan_kg(weight_kg, height_cm, sex):
    male = 1.10 * weight_kg - 120.0 * (weight_kg / height_cm) ** 2
    female = 1.07 * weight_kg - 148.0 * (weight_kg / height_cm) ** 2
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


def lbm_janma_kg(weight_kg, height_cm, sex):
    bmi = weight_kg / ((height_cm / 100.0) ** 2)
    male = 9270.0 * weight_kg / (6680.0 + 216.0 * bmi)
    female = 9270.0 * weight_kg / (8780.0 + 244.0 * bmi)
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


def ibw_kg(height_cm, sex):
    male = 48.0 + 1.06 * (height_cm - 152.0)
    female = 45.5 + 0.91 * (height_cm - 152.0)
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


def bsa_cm2(weight_kg, height_cm):
    bsa_m2 = 0.007184 * (height_cm ** 0.725) * (weight_kg ** 0.425)
    return bsa_m2 * 10000.0


def convert_bqml_to_suvbw(datasets, values):
    ds0 = datasets[0]
    weight_g = get_weight_g(ds0)
    dose_bq_admin = get_total_dose_bq(ds0)
    half_life_s = get_half_life_s(ds0)
    lam = math.log(2.0) / half_life_s
    decay_correction = str(getattr(ds0, "DecayCorrection", "") or "").upper().strip()

    if decay_correction == "ADMIN":
        return values * (weight_g / dose_bq_admin)

    if decay_correction == "START":
        tref = get_reference_time_start(ds0)
        tadm = get_radiopharm_start_datetime(ds0)
        delta_s = (tref - tadm).total_seconds()
        dose_at_ref = dose_bq_admin * math.exp(-lam * delta_s)
        return values * (weight_g / dose_at_ref)

    if decay_correction == "NONE":
        tadm = get_radiopharm_start_datetime(ds0)
        out = np.empty_like(values, dtype=np.float64)

        for z, ds in enumerate(datasets):
            acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
            if acq_dt is None:
                acq_dt = combine_da_tm(getattr(ds, "AcquisitionDate", None), getattr(ds, "AcquisitionTime", None))
            if acq_dt is None:
                raise ValueError("Missing acquisition time for DecayCorrection=NONE")

            frame_duration_ms = safe_float(getattr(ds, "ActualFrameDuration", None))
            if frame_duration_ms is None:
                raise ValueError("Missing ActualFrameDuration for DecayCorrection=NONE")

            T = frame_duration_ms / 1000.0
            factor = (
                weight_g / dose_bq_admin
                * (lam * T) / (1.0 - math.exp(-lam * T))
                * math.exp(lam * (acq_dt - tadm).total_seconds())
            )
            out[z] = values[z] * factor
        return out

    raise NotImplementedError(f"Unsupported DecayCorrection: {decay_correction}")


def convert_gml_to_suvbw(ds0, values):
    suv_type = str(getattr(ds0, "SUVType", "") or "").upper().strip()
    if suv_type == "":
        suv_type = "BW"

    if suv_type == "BW":
        return values

    weight_kg = get_weight_kg(ds0)
    height_cm = get_height_cm(ds0)
    sex = get_patient_sex(ds0)

    if suv_type == "LBM":
        factor = lbm_morgan_kg(weight_kg, height_cm, sex)
    elif suv_type == "LBMJAMES128":
        factor = lbm_james128_kg(weight_kg, height_cm, sex)
    elif suv_type == "LBMJANMA":
        factor = lbm_janma_kg(weight_kg, height_cm, sex)
    elif suv_type == "IBW":
        factor = ibw_kg(height_cm, sex)
    else:
        raise NotImplementedError(f"Unsupported SUVType in GML: {suv_type}")

    return values * (weight_kg / factor)


def convert_cm2ml_to_suvbw(ds0, values):
    weight_kg = get_weight_kg(ds0)
    height_cm = get_height_cm(ds0)
    bsa = bsa_cm2(weight_kg, height_cm)
    return values * (weight_kg / bsa) * 10000.0


def convert_cnts_to_suvbw(datasets, values):
    ds0 = datasets[0]
    suv_sf_tag = ds0.get((0x7053, 0x1000))
    ac_sf_tag = ds0.get((0x7053, 0x1009))

    suv_sf = safe_float(suv_sf_tag.value if suv_sf_tag is not None else None)
    ac_sf = safe_float(ac_sf_tag.value if ac_sf_tag is not None else None)

    if suv_sf is not None and suv_sf != 0:
        return values * suv_sf

    if ac_sf is not None and ac_sf != 0:
        bqml = values * ac_sf
        return convert_bqml_to_suvbw(datasets, bqml)

    raise NotImplementedError("CNTS without Philips scale factors is not supported.")


def convert_to_suvbw(datasets, volume, slopes, intercepts):
    ds0 = datasets[0]

    if np.any(np.abs(intercepts) > 1e-12):
        raise ValueError("RescaleIntercept is not zero. Excluding dataset.")

    values = apply_rescale(volume, slopes, intercepts)
    units = str(getattr(ds0, "Units", "") or "").upper().strip()

    if units == "BQML":
        return convert_bqml_to_suvbw(datasets, values)
    if units == "GML":
        return convert_gml_to_suvbw(ds0, values)
    if units == "CM2ML":
        return convert_cm2ml_to_suvbw(ds0, values)
    if units == "CNTS":
        return convert_cnts_to_suvbw(datasets, values)

    raise NotImplementedError(f"Unsupported Units: {units}")


def load_mask(mask_path, pet_shape=None):
    img = nib.load(mask_path)
    mask = img.get_fdata(dtype=np.float64) > 0

    if pet_shape is None:
        return mask

    if mask.shape == pet_shape:
        return mask

    candidate1 = np.transpose(mask, (2, 1, 0))
    if candidate1.shape == pet_shape:
        return candidate1

    candidate2 = np.transpose(mask, (2, 0, 1))
    if candidate2.shape == pet_shape:
        return candidate2

    raise ValueError(f"Shape mismatch: PET {pet_shape} vs mask {mask.shape}")


def calculate_ibsi_values(suv_volume, mask):
    if suv_volume.shape != mask.shape:
        raise ValueError(f"Shape mismatch: PET {suv_volume.shape} vs mask {mask.shape}")

    vals = suv_volume[mask]
    if vals.size == 0:
        raise ValueError("Mask is empty")

    return {
        "SUVmin": round(float(np.min(vals)), 2),
        "SUVmedian": round(float(np.median(vals)), 2),
        "SUVmax": round(float(np.max(vals)), 2),
        "SUVmean": round(float(np.mean(vals)), 2),
    }


def process_case(case_dir):
    case_name = os.path.basename(case_dir.rstrip(os.sep))
    pt_dir = os.path.join(case_dir, "PT")
    mask_dir = os.path.join(case_dir, "mask")

    mask_path = find_first_nii_gz(mask_dir)
    datasets, volume, slopes, intercepts = load_pet_series(pt_dir)
    suv = convert_to_suvbw(datasets, volume, slopes, intercepts)
    mask = load_mask(mask_path, pet_shape=suv.shape)
    metrics = calculate_ibsi_values(suv, mask)

    return {
        "case": case_name,
        "mask_file": os.path.basename(mask_path),
        "SUVmin": metrics["SUVmin"],
        "SUVmedian": metrics["SUVmedian"],
        "SUVmax": metrics["SUVmax"],
        "SUVmean": metrics["SUVmean"],
    }


def run_all_cases(root_dir, output_csv):
    results = []
    for case_dir in find_case_dirs(root_dir):
        case_name = os.path.basename(case_dir)
        try:
            row = process_case(case_dir)
            row["reason of error"] = ""
            results.append(row)
            print(
                f"{case_name}: "
                f"SUVmin={row['SUVmin']:.2f}, "
                f"SUVmedian={row['SUVmedian']:.2f}, "
                f"SUVmax={row['SUVmax']:.2f}"
            )
        except Exception as e:
            results.append({
                "case": case_name,
                "mask_file": "",
                "SUVmin": "ERROR",
                "SUVmedian": "ERROR",
                "SUVmax": "ERROR",
                "reason of error": str(e),
                "SUVmean": "ERROR",
            })
            print(f"{case_name}: FAILED -> {e}")

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["case", "mask_file", "SUVmin", "SUVmedian", "SUVmax", "reason of error", "SUVmean"]
        )
        writer.writeheader()
        writer.writerows(results)

    return pd.DataFrame(results)

ROOT_DIR = "/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Real Patients"
OUTPUT_CSV = "/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Results/newResult 7.csv"

for name in sorted(os.listdir(ROOT_DIR)):
    case_path = os.path.join(ROOT_DIR, name)
    if os.path.isdir(case_path):
        print(name, "->", os.listdir(case_path))



df = run_all_cases(ROOT_DIR, OUTPUT_CSV)
df

validation_01 -> ['attribution_ACRIN-FLT-Breast.md', 'mask', 'PT']
validation_02 -> ['attribution_ACRIN-FLT-Breast.md', 'mask', 'PT']
validation_03 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_04 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_05 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_06 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_07 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_08 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_09 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_10 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_11 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_12 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_13 -> ['attribution_QIN-BREAST.md', 'mask', 'PT']
validation_14 -> ['attribution_NSCLC-Radiogenomics.md', 'PT', 'mask']
validation_15 -> ['attribution_Soft

,case,mask_file,SUVmin,SUVmedian,SUVmax,SUVmean,reason of error
0,validation_01,mask.nii.gz,3.93,4.98,5.64,4.95,
1,validation_02,mask.nii.gz,3.76,5.3,7.03,5.31,
2,validation_03,mask.nii.gz,1.15,1.8,2.91,1.82,
3,validation_04,mask.nii.gz,2.36,2.79,3.34,2.8,
4,validation_05,mask.nii.gz,1.23,1.86,3.08,1.9,
5,validation_06,,ERROR,ERROR,ERROR,ERROR,Missing or zero PatientWeight
6,validation_07,mask.nii.gz,1.52,2.53,4.15,2.57,
7,validation_08,,ERROR,ERROR,ERROR,ERROR,Missing or zero PatientWeight
8,validation_09,mask.nii.gz,1.26,1.54,1.84,1.53,
9,validation_10,mask.nii.gz,1.31,2.12,3.3,2.13,
